In [1]:
import pandas as pd
import numpy as np


In [2]:
ratings = pd.read_csv('../data/ml-32m/ratings.csv')

In [3]:
print(f"Dataset Size: {ratings.memory_usage(index=True).sum() / 1024**2} MB")

Dataset Size: 976.5688514709473 MB


In [4]:
ratings.dtypes

userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

In [5]:
ratings.userId.max()

np.int64(200948)

In [6]:
# Set userId to be uint32 to save memory
ratings['userId'] = ratings['userId'].astype(np.uint32)

Ratings

In [7]:
ratings.rating.min()

np.float64(0.5)

In [8]:
ratings.rating.max()

np.float64(5.0)

In [9]:
# Set rating to be half since it can only take values from 0.5 to 5.0 in 0.5 increments
ratings['rating'] = ratings['rating'].astype(np.float16)

MovieId

In [10]:
print(ratings.movieId.max())

292757


In [11]:
# Set movieId to be uint32 to save memory
ratings['movieId'] = ratings['movieId'].astype(np.uint32)

Timestamp

In [12]:
print(ratings.timestamp.max())

1697164147


In [13]:
# Set timestamp to be uint32 to save memory
ratings['timestamp'] = ratings['timestamp'].astype(np.uint32)

### Size after setting correct dtypes

In [14]:
print(f"Dataset Size: {ratings.memory_usage(index=True).sum() / 1024**2} MB")

Dataset Size: 427.2489433288574 MB


# Make User-Item Matrix

Create a `scipy` sparse matrix to save spaces.
- Using Pandas `pivot` requires ~32GB of RAM
- `scipy` saves only the nonzero entries
- Matrix can be saved as `.npz` file

In [15]:
from scipy.sparse import csr_matrix

# Map userId and movieId to contiguous 0-based indices
user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
movie_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)}

row = ratings['userId'].map(user_to_idx).values
col = ratings['movieId'].map(movie_to_idx).values
data = np.ones(len(ratings), dtype=np.int8)  # implicit feedback: 1 = rated

user_item_matrix = csr_matrix((data, (row, col)), shape=(len(user_ids), len(movie_ids)))

print(f"Shape: {user_item_matrix.shape}")
print(f"Non-zero entries: {user_item_matrix.nnz:,}")
print(f"Sparsity: {1 - user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.6f}")
print(f"Memory usage: {(user_item_matrix.data.nbytes + user_item_matrix.indices.nbytes + user_item_matrix.indptr.nbytes) / 1024**2:.2f} MB")

Shape: (200948, 84432)
Non-zero entries: 32,000,204
Sparsity: 0.998114
Memory usage: 153.36 MB


Save matrix

In [17]:
from scipy.sparse import save_npz
save_npz('../data/ml-32m/implicit_binary_user_item_matrix.npz', user_item_matrix)